# P3 Final Submission Pipeline

이 Notebook은 **배포된 `data/` 원본 파일만으로 최종 제출 결과를 재생성**한다.

최종 모델 구조는 다음과 같다.

- **+3 h**: `Core130 + AtmosShape19 = AtmosShape149`, CatBoost Depth 4, fixed 10-seed arithmetic mean
- **+6 / +9 / +12 / +18 / +24 h**: `CoreAtmosCompact130`, CatBoost Depth 4, fixed 10-seed arithmetic mean
- Target: `ΔHs = Hs(t + lead) - Hs(t)`
- Station encoding: `G-ORS / I-ORS / S-ORS` one-hot
- 최종 생성 파일: `final_models.pkl`, `gate_params_final.pkl`, `submission.csv`

`gate_params_final.pkl`은 **고정 lead-to-branch routing 및 feature/configuration metadata**를 저장한다.


In [1]:
# ==== 모듈 import 및 제출 파이프라인 전역 설정 ==== #

from pathlib import Path
import pickle
import hashlib
import gc
import warnings

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
import catboost

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------
# 경로 설정
# ---------------------------------------------------------------------
# pipeline_final.ipynb를 저장소(P3/) 루트에서 실행하는 것을 기준으로 한다.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

FINAL_MODELS_FILE = PROJECT_ROOT / "final_models.pkl"
GATE_PARAMS_FILE = PROJECT_ROOT / "gate_params_final.pkl"
SUBMISSION_FILE = PROJECT_ROOT / "submission.csv"


# ---------------------------------------------------------------------
# 대회 / 시계열 구조
# ---------------------------------------------------------------------
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
TARGET_COLUMNS = [f"hs_lead_{lead_h}h" for lead_h in LEAD_HOURS]

WAVE_POINTS_PER_HOUR = 3      # 20분 간격
ATMOS_POINTS_PER_HOUR = 6     # 10분 간격

HS_LAG_HOURS = [1, 3, 6, 12, 24, 48]
HS_ROLL_HOURS = [1, 3, 6, 12, 24, 48]
WVDIR_LAG_HOURS = [3, 6, 12]

ATMOS_LAG_HOURS = [1, 3, 6, 12, 24]
ATMOS_ROLL_HOURS = [1, 3, 6, 12, 24, 48]
WIND_DIR_LAG_HOURS = [3, 6, 12]

STATION_ORDER = ["G-ORS", "I-ORS", "S-ORS"]
KEY_COLUMNS = ["case_id", "station", "lead_h"]


# ---------------------------------------------------------------------
# 최종 모델 Multi-Seed 설정
# 사전에 고정했던 10개 seed를 그대로 사용한다.
# ---------------------------------------------------------------------
MULTISEED_LIST = [
    11, 29, 47, 71, 101,
    137, 173, 211, 251, 307
]


# ---------------------------------------------------------------------
# 최종 CatBoost 설정
# 사용한 Depth-4 recipe와 동일하다.
# ---------------------------------------------------------------------
CATBOOST_BASE_PARAMS = {
    "iterations": 450,
    "learning_rate": 0.03,
    "depth": 4,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "loss_function": "RMSE",
    "verbose": False,
    "allow_writing_files": False,
}

EXPECTED_CATBOOST_VERSION = "1.2.10"


# ---------------------------------------------------------------------
# 필수 입력 파일
# ---------------------------------------------------------------------
REQUIRED_DATA_FILES = {
    "train_wave": DATA_DIR / "train_wave.csv",
    "train_atmos": DATA_DIR / "train_atmos.csv",
    "test_context": DATA_DIR / "test_context.parquet",
    "test_index": DATA_DIR / "test_index.csv",
    "sample_submission": DATA_DIR / "sample_submission.csv",
}

missing_files = [
    str(path)
    for path in REQUIRED_DATA_FILES.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "다음 배포 데이터 파일을 찾지 못했습니다.\n"
        + "\n".join(missing_files)
    )

print("=" * 120)
print("P3 FINAL REPRODUCIBILITY PIPELINE")
print("=" * 120)
print(f"Project root     : {PROJECT_ROOT}")
print(f"Data directory   : {DATA_DIR}")
print(f"CatBoost version : {catboost.__version__}")

if catboost.__version__ != EXPECTED_CATBOOST_VERSION:
    print(
        "[주의] exact reproduction은 "
        f"catboost=={EXPECTED_CATBOOST_VERSION}에서 확인되었습니다. "
        "버전이 다르면 극미한 수치 차이가 발생할 수 있습니다."
    )

print("[완료] 전역 설정 및 입력 파일 확인 완료")


P3 FINAL REPRODUCIBILITY PIPELINE
Project root     : c:\Users\신현진\Desktop\P3
Data directory   : c:\Users\신현진\Desktop\P3\data
CatBoost version : 1.2.10
[완료] 전역 설정 및 입력 파일 확인 완료


In [2]:
def convert_time_column(
    df,
    column="time"
):
    """
    시간 열을 timezone-aware datetime으로 변환한다.

    기존 자료의 +09:00 정보를 보존하고,
    merge 시 시간 dtype 불일치가 발생하지 않도록 한다.
    """

    result = df.copy()

    result[column] = pd.to_datetime(
        result[column],
        utc=True
    ).dt.tz_convert(
        "Asia/Seoul"
    )

    return result

# ==== 배포 Raw Data 로드 및 기본 무결성 확인 ==== #

try:
    train_wave = pd.read_csv(REQUIRED_DATA_FILES["train_wave"])
    train_atmos = pd.read_csv(REQUIRED_DATA_FILES["train_atmos"])
    test_context = pd.read_parquet(REQUIRED_DATA_FILES["test_context"])
    test_index = pd.read_csv(REQUIRED_DATA_FILES["test_index"])
    sample_submission = pd.read_csv(REQUIRED_DATA_FILES["sample_submission"])

    train_wave = convert_time_column(train_wave)
    train_atmos = convert_time_column(train_atmos)

    train_wave = (
        train_wave
        .sort_values(["station", "time"])
        .reset_index(drop=True)
    )

    train_atmos = (
        train_atmos
        .sort_values(["station", "time"])
        .reset_index(drop=True)
    )

    test_context = (
        test_context
        .sort_values(["case_id", "step_minute"])
        .reset_index(drop=True)
    )

    required_wave_columns = {"station", "time", "hs", "tp", "hmax", "wvdir"}
    required_atmos_columns = {
        "station", "time", "wspd", "gust", "wdir", "airt", "relh", "caph"
    }
    required_test_context_columns = {
        "case_id", "station", "step_minute",
        "hs", "tp", "hmax", "wvdir",
        "wspd", "gust", "wdir", "airt", "relh", "caph"
    }
    required_submission_columns = {"case_id", "station", "lead_h", "hs_pred"}

    for label, df, required_columns in [
        ("train_wave", train_wave, required_wave_columns),
        ("train_atmos", train_atmos, required_atmos_columns),
        ("test_context", test_context, required_test_context_columns),
        ("sample_submission", sample_submission, required_submission_columns),
    ]:
        missing = sorted(required_columns - set(df.columns))
        if missing:
            raise KeyError(f"{label}에 필요한 열이 없습니다: {missing}")

    print(f"train_wave       : {train_wave.shape}")
    print(f"train_atmos      : {train_atmos.shape}")
    print(f"test_context     : {test_context.shape}")
    print(f"test_index       : {test_index.shape}")
    print(f"sample_submission: {sample_submission.shape}")

    print("[완료] Raw Data Load 및 Schema 확인 완료")

except Exception as e:
    print(f"[실패] Raw Data Load/검증 중 오류 발생: {e}")
    raise


train_wave       : (118152, 6)
train_atmos      : (130896, 8)
test_context     : (57800, 13)
test_index       : (1200, 3)
sample_submission: (1200, 4)
[완료] Raw Data Load 및 Schema 확인 완료


In [3]:
# ==== Core Feature 생성 함수 정의 ==== #

def build_train_wave_features(
    wave
):
    """
    연속적인 Train wave 시계열로부터
    Wave feature를 생성한다.

    구성
    ----
    Hs Feature       : 55
    Wave Direction   : 11
    총               : 66
    """

    df = (
        wave[
            [
                "station",
                "time",
                "hs",
                "wvdir"
            ]
        ]
        .copy()
        .sort_values(
            [
                "station",
                "time"
            ]
        )
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------
    # Hs 현재값
    # -------------------------------------------------------------

    df[
        "hs_current"
    ] = df[
        "hs"
    ]


    # -------------------------------------------------------------
    # Hs Lag / Change / Rate
    # -------------------------------------------------------------

    for lag_h in HS_LAG_HOURS:

        lag_rows = (
            lag_h
            * WAVE_POINTS_PER_HOUR
        )

        lag_value = (
            df.groupby(
                "station"
            )[
                "hs"
            ]
            .shift(
                lag_rows
            )
        )

        df[
            f"hs_lag_{lag_h}h"
        ] = lag_value

        df[
            f"hs_change_{lag_h}h"
        ] = (
            df[
                "hs"
            ]
            - lag_value
        )

        df[
            f"hs_rate_{lag_h}h"
        ] = (
            df[
                f"hs_change_{lag_h}h"
            ]
            / lag_h
        )


    # -------------------------------------------------------------
    # Hs Rolling Statistics
    # -------------------------------------------------------------

    for window_h in HS_ROLL_HOURS:

        window_points = (
            window_h
            * WAVE_POINTS_PER_HOUR
            + 1
        )

        min_valid_points = max(
            2,
            int(
                np.ceil(
                    0.5
                    * window_points
                )
            )
        )

        grouped = (
            df.groupby(
                "station"
            )[
                "hs"
            ]
        )


        df[
            f"hs_mean_{window_h}h"
        ] = grouped.transform(

            lambda s:
            s.rolling(
                window=
                    window_points,
                min_periods=
                    min_valid_points
            ).mean()
        )


        df[
            f"hs_std_{window_h}h"
        ] = grouped.transform(

            lambda s:
            s.rolling(
                window=
                    window_points,
                min_periods=
                    min_valid_points
            ).std()
        )


        df[
            f"hs_min_{window_h}h"
        ] = grouped.transform(

            lambda s:
            s.rolling(
                window=
                    window_points,
                min_periods=
                    min_valid_points
            ).min()
        )


        df[
            f"hs_max_{window_h}h"
        ] = grouped.transform(

            lambda s:
            s.rolling(
                window=
                    window_points,
                min_periods=
                    min_valid_points
            ).max()
        )


        df[
            f"hs_valid_ratio_{window_h}h"
        ] = grouped.transform(

            lambda s:
            s.notna()
            .rolling(
                window=
                    window_points,
                min_periods=1
            )
            .mean()
        )


        df[
            f"hs_anom_mean_{window_h}h"
        ] = (
            df[
                "hs"
            ]
            - df[
                f"hs_mean_{window_h}h"
            ]
        )


    # -------------------------------------------------------------
    # Wave Direction 현재값
    # -------------------------------------------------------------

    df[
        "wvdir_sin_current"
    ] = np.sin(
        np.deg2rad(
            df[
                "wvdir"
            ]
        )
    )

    df[
        "wvdir_cos_current"
    ] = np.cos(
        np.deg2rad(
            df[
                "wvdir"
            ]
        )
    )


    # -------------------------------------------------------------
    # Wave Direction Lag
    # -------------------------------------------------------------

    for lag_h in WVDIR_LAG_HOURS:

        lag_rows = (
            lag_h
            * WAVE_POINTS_PER_HOUR
        )

        wvdir_lag = (
            df.groupby(
                "station"
            )[
                "wvdir"
            ]
            .shift(
                lag_rows
            )
        )


        df[
            f"wvdir_sin_lag_{lag_h}h"
        ] = np.sin(
            np.deg2rad(
                wvdir_lag
            )
        )

        df[
            f"wvdir_cos_lag_{lag_h}h"
        ] = np.cos(
            np.deg2rad(
                wvdir_lag
            )
        )

        df[
            f"wvdir_alignment_{lag_h}h"
        ] = np.cos(
            np.deg2rad(
                df[
                    "wvdir"
                ]
                - wvdir_lag
            )
        )


    # -------------------------------------------------------------
    # Feature 이름 정의
    # -------------------------------------------------------------

    hs_features = [
        "hs_current"
    ]


    for lag_h in HS_LAG_HOURS:

        hs_features.extend([
            f"hs_lag_{lag_h}h",
            f"hs_change_{lag_h}h",
            f"hs_rate_{lag_h}h"
        ])


    for window_h in HS_ROLL_HOURS:

        hs_features.extend([
            f"hs_mean_{window_h}h",
            f"hs_std_{window_h}h",
            f"hs_min_{window_h}h",
            f"hs_max_{window_h}h",
            f"hs_valid_ratio_{window_h}h",
            f"hs_anom_mean_{window_h}h"
        ])


    wvdir_features = [
        "wvdir_sin_current",
        "wvdir_cos_current"
    ]


    for lag_h in WVDIR_LAG_HOURS:

        wvdir_features.extend([
            f"wvdir_sin_lag_{lag_h}h",
            f"wvdir_cos_lag_{lag_h}h",
            f"wvdir_alignment_{lag_h}h"
        ])


    return (
        df[
            [
                "station",
                "time"
            ]
            + hs_features
            + wvdir_features
        ],
        hs_features,
        wvdir_features
    )

def build_train_atmos_features(
    atmos
):
    """
    Train의 10분 간격 대기 연속 시계열에서
    Atmospheric Feature 127개를 생성한다.
    """

    df = (
        atmos[
            [
                "station",
                "time",
                "wspd",
                "gust",
                "wdir",
                "caph"
            ]
        ]
        .copy()
        .sort_values(
            [
                "station",
                "time"
            ]
        )
        .reset_index(drop=True)
    )


    # -------------------------------------------------------------
    # 현재 상태
    # -------------------------------------------------------------

    df[
        "wspd_current"
    ] = df[
        "wspd"
    ]

    df[
        "gust_current"
    ] = df[
        "gust"
    ]

    df[
        "caph_current"
    ] = df[
        "caph"
    ]

    df[
        "gust_excess_current"
    ] = (
        df[
            "gust"
        ]
        - df[
            "wspd"
        ]
    )


    df[
        "wdir_sin_current"
    ] = np.sin(
        np.deg2rad(
            df[
                "wdir"
            ]
        )
    )

    df[
        "wdir_cos_current"
    ] = np.cos(
        np.deg2rad(
            df[
                "wdir"
            ]
        )
    )


    # -------------------------------------------------------------
    # Wind magnitude / Pressure Lag
    # -------------------------------------------------------------

    for lag_h in ATMOS_LAG_HOURS:

        lag_rows = (
            lag_h
            * ATMOS_POINTS_PER_HOUR
        )


        for variable in [
            "wspd",
            "gust",
            "caph"
        ]:

            lag_value = (
                df.groupby(
                    "station"
                )[
                    variable
                ]
                .shift(
                    lag_rows
                )
            )


            df[
                f"{variable}_lag_{lag_h}h"
            ] = lag_value


            df[
                f"{variable}_change_{lag_h}h"
            ] = (
                df[
                    variable
                ]
                - lag_value
            )


            df[
                f"{variable}_rate_{lag_h}h"
            ] = (
                df[
                    f"{variable}_change_{lag_h}h"
                ]
                / lag_h
            )


    # -------------------------------------------------------------
    # Wind Direction Lag
    # -------------------------------------------------------------

    for lag_h in WIND_DIR_LAG_HOURS:

        lag_rows = (
            lag_h
            * ATMOS_POINTS_PER_HOUR
        )


        wdir_lag = (
            df.groupby(
                "station"
            )[
                "wdir"
            ]
            .shift(
                lag_rows
            )
        )


        df[
            f"wdir_sin_lag_{lag_h}h"
        ] = np.sin(
            np.deg2rad(
                wdir_lag
            )
        )


        df[
            f"wdir_cos_lag_{lag_h}h"
        ] = np.cos(
            np.deg2rad(
                wdir_lag
            )
        )


        df[
            f"wdir_alignment_{lag_h}h"
        ] = np.cos(
            np.deg2rad(
                df[
                    "wdir"
                ]
                - wdir_lag
            )
        )


    # -------------------------------------------------------------
    # Rolling Statistics
    # -------------------------------------------------------------

    for window_h in ATMOS_ROLL_HOURS:

        window_points = (
            window_h
            * ATMOS_POINTS_PER_HOUR
            + 1
        )


        min_valid_points = max(
            2,
            int(
                np.ceil(
                    window_points
                    * 0.5
                )
            )
        )


        for variable in [
            "wspd",
            "gust"
        ]:

            grouped = (
                df.groupby(
                    "station"
                )[
                    variable
                ]
            )


            df[
                f"{variable}_mean_{window_h}h"
            ] = grouped.transform(

                lambda s:
                s.rolling(
                    window=
                        window_points,
                    min_periods=
                        min_valid_points
                ).mean()
            )


            df[
                f"{variable}_std_{window_h}h"
            ] = grouped.transform(

                lambda s:
                s.rolling(
                    window=
                        window_points,
                    min_periods=
                        min_valid_points
                ).std()
            )


            df[
                f"{variable}_max_{window_h}h"
            ] = grouped.transform(

                lambda s:
                s.rolling(
                    window=
                        window_points,
                    min_periods=
                        min_valid_points
                ).max()
            )


            df[
                f"{variable}_valid_ratio_{window_h}h"
            ] = grouped.transform(

                lambda s:
                s.notna()
                .rolling(
                    window=
                        window_points,
                    min_periods=1
                )
                .mean()
            )


        # ---------------------------------------------------------
        # Pressure
        # ---------------------------------------------------------

        caph_grouped = (
            df.groupby(
                "station"
            )[
                "caph"
            ]
        )


        df[
            f"caph_mean_{window_h}h"
        ] = caph_grouped.transform(

            lambda s:
            s.rolling(
                window=
                    window_points,
                min_periods=
                    min_valid_points
            ).mean()
        )


        df[
            f"caph_std_{window_h}h"
        ] = caph_grouped.transform(

            lambda s:
            s.rolling(
                window=
                    window_points,
                min_periods=
                    min_valid_points
            ).std()
        )


        df[
            f"caph_valid_ratio_{window_h}h"
        ] = caph_grouped.transform(

            lambda s:
            s.notna()
            .rolling(
                window=
                    window_points,
                min_periods=1
            )
            .mean()
        )


    # -------------------------------------------------------------
    # 현재 Atmos Availability
    # -------------------------------------------------------------

    df[
        "atmos_current_available"
    ] = (
        df[
            [
                "wspd",
                "gust",
                "wdir",
                "caph"
            ]
        ]
        .notna()
        .any(axis=1)
        .astype(int)
    )


    # -------------------------------------------------------------
    # Atmos Feature List
    # -------------------------------------------------------------

    feature_columns = [

        c
        for c in df.columns

        if c not in [
            "station",
            "time",
            "wspd",
            "gust",
            "wdir",
            "caph"
        ]
    ]


    # -------------------------------------------------------------
    # Event-pool 품질 진단용 helper
    # 모델 Feature에는 포함하지 않는다.
    # -------------------------------------------------------------

    window_points_48h = (
        48
        * ATMOS_POINTS_PER_HOUR
        + 1
    )


    df[
        "_wdir_valid_ratio_48h"
    ] = (
        df.groupby(
            "station"
        )[
            "wdir"
        ]
        .transform(

            lambda s:
            s.notna()
            .rolling(
                window=
                    window_points_48h,
                min_periods=1
            )
            .mean()
        )
    )


    df[
        "_atmos_station_row"
    ] = (
        df.groupby(
            "station"
        )
        .cumcount()
    )


    df[
        "_full_atmos_48h"
    ] = (
        df[
            "_atmos_station_row"
        ]
        .ge(
            48
            * ATMOS_POINTS_PER_HOUR
        )
        .astype(int)
    )


    return (
        df[
            [
                "station",
                "time"
            ]
            + feature_columns
            + [
                "_wdir_valid_ratio_48h",
                "_full_atmos_48h"
            ]
        ],
        feature_columns
    )

def add_wind_wave_interactions(
    df
):
    """
    Wind와 Wave propagation direction의 상대방향을 이용하여
    16개의 interaction feature를 생성한다.

    Current / 3h / 6h / 12h 각각:
    - alignment
    - cross component
    - parallel wind speed
    - parallel gust
    """

    result = df.copy()

    interaction_features = []


    for lag_h in [
        None,
        3,
        6,
        12
    ]:

        if lag_h is None:

            suffix = "current"

            wind_sin = (
                result[
                    "wdir_sin_current"
                ]
            )

            wind_cos = (
                result[
                    "wdir_cos_current"
                ]
            )

            wave_sin = (
                result[
                    "wvdir_sin_current"
                ]
            )

            wave_cos = (
                result[
                    "wvdir_cos_current"
                ]
            )

            wspd = (
                result[
                    "wspd_current"
                ]
            )

            gust = (
                result[
                    "gust_current"
                ]
            )


        else:

            suffix = (
                f"{lag_h}h"
            )

            wind_sin = (
                result[
                    f"wdir_sin_lag_{lag_h}h"
                ]
            )

            wind_cos = (
                result[
                    f"wdir_cos_lag_{lag_h}h"
                ]
            )

            wave_sin = (
                result[
                    f"wvdir_sin_lag_{lag_h}h"
                ]
            )

            wave_cos = (
                result[
                    f"wvdir_cos_lag_{lag_h}h"
                ]
            )

            wspd = (
                result[
                    f"wspd_lag_{lag_h}h"
                ]
            )

            gust = (
                result[
                    f"gust_lag_{lag_h}h"
                ]
            )


        # ---------------------------------------------------------
        # cos(wind direction - wave direction)
        # ---------------------------------------------------------

        alignment = (

            wind_cos
            * wave_cos

            +

            wind_sin
            * wave_sin
        )


        # ---------------------------------------------------------
        # sin(wind direction - wave direction)
        # signed cross component
        # ---------------------------------------------------------

        cross = (

            wind_sin
            * wave_cos

            -

            wind_cos
            * wave_sin
        )


        align_name = (
            f"wind_wave_alignment_{suffix}"
        )

        cross_name = (
            f"wind_wave_cross_{suffix}"
        )

        wspd_name = (
            f"wspd_parallel_wave_{suffix}"
        )

        gust_name = (
            f"gust_parallel_wave_{suffix}"
        )


        result[
            align_name
        ] = alignment

        result[
            cross_name
        ] = cross

        result[
            wspd_name
        ] = (
            wspd
            * alignment
        )

        result[
            gust_name
        ] = (
            gust
            * alignment
        )


        interaction_features.extend([
            align_name,
            cross_name,
            wspd_name,
            gust_name
        ])


    return (
        result,
        interaction_features
    )

def build_test_wave_features(
    context
):
    """
    각 Test case의 -48h ~ 0h context로부터
    Train과 동일한 66개 Wave feature set feature를 생성한다.
    """

    rows = []


    for case_id, g in (
        context.groupby(
            "case_id",
            sort=False
        )
    ):

        g = (
            g
            .sort_values(
                "step_minute"
            )
            .copy()
        )


        station = (
            g[
                "station"
            ]
            .iloc[0]
        )


        # Wave는 20분 grid
        wave_g = (
            g[
                g[
                    "step_minute"
                ]
                .mod(20)
                .eq(0)
            ]
            .copy()
        )


        current_rows = (
            wave_g[
                wave_g[
                    "step_minute"
                ]
                .eq(0)
            ]
        )


        if len(
            current_rows
        ) != 1:

            raise ValueError(
                f"{case_id}: "
                "step_minute=0 Wave 행 오류"
            )


        current = (
            current_rows
            .iloc[0]
        )


        row = {

            "case_id":
                case_id,

            "station":
                station,

            "hs_current":
                current[
                    "hs"
                ]
        }


        # ---------------------------------------------------------
        # Hs Lag
        # ---------------------------------------------------------

        for lag_h in HS_LAG_HOURS:

            target_step = (
                -lag_h
                * 60
            )


            lag_rows = (
                wave_g[
                    wave_g[
                        "step_minute"
                    ]
                    .eq(
                        target_step
                    )
                ]
            )


            lag_value = (

                lag_rows[
                    "hs"
                ]
                .iloc[0]

                if len(
                    lag_rows
                ) == 1

                else np.nan
            )


            row[
                f"hs_lag_{lag_h}h"
            ] = lag_value


            if (
                pd.notna(
                    current[
                        "hs"
                    ]
                )
                and
                pd.notna(
                    lag_value
                )
            ):

                change = (
                    current[
                        "hs"
                    ]
                    - lag_value
                )

            else:

                change = np.nan


            row[
                f"hs_change_{lag_h}h"
            ] = change


            row[
                f"hs_rate_{lag_h}h"
            ] = (

                change
                / lag_h

                if pd.notna(
                    change
                )

                else np.nan
            )


        # ---------------------------------------------------------
        # Hs Rolling
        # ---------------------------------------------------------

        for window_h in HS_ROLL_HOURS:

            window_data = (
                wave_g.loc[
                    (
                        wave_g[
                            "step_minute"
                        ]
                        >= -window_h * 60
                    )
                    &
                    (
                        wave_g[
                            "step_minute"
                        ]
                        <= 0
                    ),
                    "hs"
                ]
            )


            expected_points = (
                window_h
                * WAVE_POINTS_PER_HOUR
                + 1
            )


            min_valid_points = max(
                2,
                int(
                    np.ceil(
                        expected_points
                        * 0.5
                    )
                )
            )


            valid_count = int(
                window_data
                .notna()
                .sum()
            )


            row[
                f"hs_valid_ratio_{window_h}h"
            ] = (
                valid_count
                / expected_points
            )


            if (
                valid_count
                >= min_valid_points
            ):

                mean_value = (
                    window_data.mean()
                )

                row[
                    f"hs_mean_{window_h}h"
                ] = mean_value

                row[
                    f"hs_std_{window_h}h"
                ] = (
                    window_data.std()
                )

                row[
                    f"hs_min_{window_h}h"
                ] = (
                    window_data.min()
                )

                row[
                    f"hs_max_{window_h}h"
                ] = (
                    window_data.max()
                )

            else:

                mean_value = np.nan

                row[
                    f"hs_mean_{window_h}h"
                ] = np.nan

                row[
                    f"hs_std_{window_h}h"
                ] = np.nan

                row[
                    f"hs_min_{window_h}h"
                ] = np.nan

                row[
                    f"hs_max_{window_h}h"
                ] = np.nan


            row[
                f"hs_anom_mean_{window_h}h"
            ] = (

                current[
                    "hs"
                ]
                - mean_value

                if (
                    pd.notna(
                        current[
                            "hs"
                        ]
                    )
                    and
                    pd.notna(
                        mean_value
                    )
                )

                else np.nan
            )


        # ---------------------------------------------------------
        # Wave Direction
        # ---------------------------------------------------------

        wvdir_current = (
            current[
                "wvdir"
            ]
        )


        row[
            "wvdir_sin_current"
        ] = (

            np.sin(
                np.deg2rad(
                    wvdir_current
                )
            )

            if pd.notna(
                wvdir_current
            )

            else np.nan
        )


        row[
            "wvdir_cos_current"
        ] = (

            np.cos(
                np.deg2rad(
                    wvdir_current
                )
            )

            if pd.notna(
                wvdir_current
            )

            else np.nan
        )


        for lag_h in WVDIR_LAG_HOURS:

            target_step = (
                -lag_h
                * 60
            )


            lag_rows = (
                wave_g[
                    wave_g[
                        "step_minute"
                    ]
                    .eq(
                        target_step
                    )
                ]
            )


            wvdir_lag = (

                lag_rows[
                    "wvdir"
                ]
                .iloc[0]

                if len(
                    lag_rows
                ) == 1

                else np.nan
            )


            row[
                f"wvdir_sin_lag_{lag_h}h"
            ] = (

                np.sin(
                    np.deg2rad(
                        wvdir_lag
                    )
                )

                if pd.notna(
                    wvdir_lag
                )

                else np.nan
            )


            row[
                f"wvdir_cos_lag_{lag_h}h"
            ] = (

                np.cos(
                    np.deg2rad(
                        wvdir_lag
                    )
                )

                if pd.notna(
                    wvdir_lag
                )

                else np.nan
            )


            row[
                f"wvdir_alignment_{lag_h}h"
            ] = (

                np.cos(
                    np.deg2rad(
                        wvdir_current
                        - wvdir_lag
                    )
                )

                if (
                    pd.notna(
                        wvdir_current
                    )
                    and
                    pd.notna(
                        wvdir_lag
                    )
                )

                else np.nan
            )


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )

def build_test_atmos_features(
    context
):
    """
    Test case별 48시간 10분 context에서
    Train과 동일한 127개의 Atmospheric Feature를 생성한다.
    """

    rows = []


    for case_id, g in (
        context.groupby(
            "case_id",
            sort=False
        )
    ):

        g = (
            g
            .sort_values(
                "step_minute"
            )
            .copy()
        )


        station = (
            g[
                "station"
            ]
            .iloc[0]
        )


        current_rows = (
            g[
                g[
                    "step_minute"
                ]
                .eq(0)
            ]
        )


        if len(
            current_rows
        ) != 1:

            raise ValueError(
                f"{case_id}: "
                "step_minute=0 Atmos 행 오류"
            )


        current = (
            current_rows
            .iloc[0]
        )


        row = {

            "case_id":
                case_id,

            "station":
                station,

            "wspd_current":
                current[
                    "wspd"
                ],

            "gust_current":
                current[
                    "gust"
                ],

            "caph_current":
                current[
                    "caph"
                ]
        }


        wspd_current = (
            current[
                "wspd"
            ]
        )

        gust_current = (
            current[
                "gust"
            ]
        )

        wdir_current = (
            current[
                "wdir"
            ]
        )


        row[
            "gust_excess_current"
        ] = (

            gust_current
            - wspd_current

            if (
                pd.notna(
                    gust_current
                )
                and
                pd.notna(
                    wspd_current
                )
            )

            else np.nan
        )


        row[
            "wdir_sin_current"
        ] = (

            np.sin(
                np.deg2rad(
                    wdir_current
                )
            )

            if pd.notna(
                wdir_current
            )

            else np.nan
        )


        row[
            "wdir_cos_current"
        ] = (

            np.cos(
                np.deg2rad(
                    wdir_current
                )
            )

            if pd.notna(
                wdir_current
            )

            else np.nan
        )


        # ---------------------------------------------------------
        # Lag / Change / Rate
        # ---------------------------------------------------------

        for lag_h in ATMOS_LAG_HOURS:

            target_step = (
                -lag_h
                * 60
            )


            lag_rows = (
                g[
                    g[
                        "step_minute"
                    ]
                    .eq(
                        target_step
                    )
                ]
            )


            lag_row = (

                lag_rows
                .iloc[0]

                if len(
                    lag_rows
                ) == 1

                else None
            )


            for variable in [
                "wspd",
                "gust",
                "caph"
            ]:

                current_value = (
                    current[
                        variable
                    ]
                )


                lag_value = (

                    lag_row[
                        variable
                    ]

                    if lag_row
                    is not None

                    else np.nan
                )


                row[
                    f"{variable}_lag_{lag_h}h"
                ] = lag_value


                change = (

                    current_value
                    - lag_value

                    if (
                        pd.notna(
                            current_value
                        )
                        and
                        pd.notna(
                            lag_value
                        )
                    )

                    else np.nan
                )


                row[
                    f"{variable}_change_{lag_h}h"
                ] = change


                row[
                    f"{variable}_rate_{lag_h}h"
                ] = (

                    change
                    / lag_h

                    if pd.notna(
                        change
                    )

                    else np.nan
                )


        # ---------------------------------------------------------
        # Wind Direction Lag
        # ---------------------------------------------------------

        for lag_h in WIND_DIR_LAG_HOURS:

            target_step = (
                -lag_h
                * 60
            )


            lag_rows = (
                g[
                    g[
                        "step_minute"
                    ]
                    .eq(
                        target_step
                    )
                ]
            )


            wdir_lag = (

                lag_rows[
                    "wdir"
                ]
                .iloc[0]

                if len(
                    lag_rows
                ) == 1

                else np.nan
            )


            row[
                f"wdir_sin_lag_{lag_h}h"
            ] = (

                np.sin(
                    np.deg2rad(
                        wdir_lag
                    )
                )

                if pd.notna(
                    wdir_lag
                )

                else np.nan
            )


            row[
                f"wdir_cos_lag_{lag_h}h"
            ] = (

                np.cos(
                    np.deg2rad(
                        wdir_lag
                    )
                )

                if pd.notna(
                    wdir_lag
                )

                else np.nan
            )


            row[
                f"wdir_alignment_{lag_h}h"
            ] = (

                np.cos(
                    np.deg2rad(
                        wdir_current
                        - wdir_lag
                    )
                )

                if (
                    pd.notna(
                        wdir_current
                    )
                    and
                    pd.notna(
                        wdir_lag
                    )
                )

                else np.nan
            )


        # ---------------------------------------------------------
        # Rolling Statistics
        # ---------------------------------------------------------

        for window_h in ATMOS_ROLL_HOURS:

            window_mask = (

                (
                    g[
                        "step_minute"
                    ]
                    >= -window_h
                    * 60
                )

                &

                (
                    g[
                        "step_minute"
                    ]
                    <= 0
                )
            )


            window_points = (
                window_h
                * ATMOS_POINTS_PER_HOUR
                + 1
            )


            min_valid_points = max(
                2,
                int(
                    np.ceil(
                        window_points
                        * 0.5
                    )
                )
            )


            for variable in [
                "wspd",
                "gust"
            ]:

                values = (
                    g.loc[
                        window_mask,
                        variable
                    ]
                )


                valid_count = int(
                    values
                    .notna()
                    .sum()
                )


                row[
                    f"{variable}_valid_ratio_{window_h}h"
                ] = (
                    valid_count
                    / window_points
                )


                if (
                    valid_count
                    >= min_valid_points
                ):

                    row[
                        f"{variable}_mean_{window_h}h"
                    ] = (
                        values.mean()
                    )

                    row[
                        f"{variable}_std_{window_h}h"
                    ] = (
                        values.std()
                    )

                    row[
                        f"{variable}_max_{window_h}h"
                    ] = (
                        values.max()
                    )

                else:

                    row[
                        f"{variable}_mean_{window_h}h"
                    ] = np.nan

                    row[
                        f"{variable}_std_{window_h}h"
                    ] = np.nan

                    row[
                        f"{variable}_max_{window_h}h"
                    ] = np.nan


            caph_values = (
                g.loc[
                    window_mask,
                    "caph"
                ]
            )


            caph_valid_count = int(
                caph_values
                .notna()
                .sum()
            )


            row[
                f"caph_valid_ratio_{window_h}h"
            ] = (
                caph_valid_count
                / window_points
            )


            if (
                caph_valid_count
                >= min_valid_points
            ):

                row[
                    f"caph_mean_{window_h}h"
                ] = (
                    caph_values.mean()
                )

                row[
                    f"caph_std_{window_h}h"
                ] = (
                    caph_values.std()
                )

            else:

                row[
                    f"caph_mean_{window_h}h"
                ] = np.nan

                row[
                    f"caph_std_{window_h}h"
                ] = np.nan


        row[
            "atmos_current_available"
        ] = int(

            pd.notna(
                current[
                    [
                        "wspd",
                        "gust",
                        "wdir",
                        "caph"
                    ]
                ]
            )
            .any()
        )


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )

print("[완료] Train/Test Feature Builder 정의 완료")


[완료] Train/Test Feature Builder 정의 완료


In [4]:
# ==== Train Candidate / Target 및 Base Feature Table 생성 ==== #

try:
    # -----------------------------------------------------------------
    # Train candidate와 6개 미래 Hs target 생성
    # -----------------------------------------------------------------
    wave_target_base = train_wave[["station", "time", "hs"]].copy()
    wave_group = wave_target_base.groupby("station", sort=False)

    for lead_h in LEAD_HOURS:
        lead_rows = lead_h * WAVE_POINTS_PER_HOUR
        wave_target_base[f"hs_lead_{lead_h}h"] = (
            wave_group["hs"].shift(-lead_rows)
        )

    wave_target_base["_station_row"] = (
        wave_target_base.groupby("station").cumcount()
    )
    wave_target_base["_station_size"] = (
        wave_target_base.groupby("station")["time"].transform("size")
    )

    context_rows = 48 * WAVE_POINTS_PER_HOUR
    max_future_rows = 24 * WAVE_POINTS_PER_HOUR

    candidate_mask = (
        wave_target_base["hs"].ge(1.5)
        & wave_target_base[TARGET_COLUMNS].notna().all(axis=1)
        & wave_target_base["_station_row"].ge(context_rows)
        & (
            wave_target_base["_station_row"] + max_future_rows
            < wave_target_base["_station_size"]
        )
    )

    train_candidates = (
        wave_target_base.loc[
            candidate_mask,
            ["station", "time"] + TARGET_COLUMNS
        ]
        .copy()
        .reset_index(drop=True)
    )

    expected_candidate_counts = {
        "G-ORS": 9893,
        "I-ORS": 7312,
        "S-ORS": 7155,
    }

    observed_counts = (
        train_candidates["station"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    if len(train_candidates) != 24360:
        raise ValueError(
            "Train candidate 수가 기존 final pipeline과 다릅니다: "
            f"{len(train_candidates):,} != 24,360"
        )

    if {
        station: int(observed_counts.get(station, 0))
        for station in STATION_ORDER
    } != expected_candidate_counts:
        raise ValueError(
            "Station별 Train candidate 수가 기존 final pipeline과 다릅니다.\n"
            f"Observed: {observed_counts}\n"
            f"Expected: {expected_candidate_counts}"
        )

    print("Train Candidate 24,360개 정확 복원 완료")


    # -----------------------------------------------------------------
    # 연속 Train 시계열에서 feature 생성
    # -----------------------------------------------------------------
    (
        train_wave_features,
        HS_FEATURES,
        WVDIR_FEATURES,
    ) = build_train_wave_features(train_wave)

    (
        train_atmos_features,
        ATMOS_FEATURES,
    ) = build_train_atmos_features(train_atmos)

    if len(HS_FEATURES) != 55:
        raise ValueError(f"Hs Feature 수 오류: {len(HS_FEATURES)}")
    if len(WVDIR_FEATURES) != 11:
        raise ValueError(f"Wave Direction Feature 수 오류: {len(WVDIR_FEATURES)}")
    if len(ATMOS_FEATURES) != 127:
        raise ValueError(f"Atmospheric Feature 수 오류: {len(ATMOS_FEATURES)}")

    train_feature_table = (
        train_candidates
        .merge(
            train_wave_features,
            on=["station", "time"],
            how="left",
            validate="one_to_one",
        )
        .merge(
            train_atmos_features,
            on=["station", "time"],
            how="left",
            validate="one_to_one",
        )
    )

    (
        train_feature_table,
        WIND_WAVE_FEATURES,
    ) = add_wind_wave_interactions(train_feature_table)

    if len(WIND_WAVE_FEATURES) != 16:
        raise ValueError(
            f"Wind-Wave Interaction Feature 수 오류: {len(WIND_WAVE_FEATURES)}"
        )

    print("Train Feature Table 생성 완료")


    # -----------------------------------------------------------------
    # Test context에서 동일한 feature 생성
    # -----------------------------------------------------------------
    test_wave_features = build_test_wave_features(test_context)
    test_atmos_features = build_test_atmos_features(test_context)

    test_feature_table = (
        test_wave_features
        .merge(
            test_atmos_features,
            on=["case_id", "station"],
            how="left",
            validate="one_to_one",
        )
    )

    (
        test_feature_table,
        test_wind_wave_features,
    ) = add_wind_wave_interactions(test_feature_table)

    if WIND_WAVE_FEATURES != test_wind_wave_features:
        raise ValueError("Train/Test Wind-Wave Feature 이름 또는 순서가 다릅니다.")

    if len(test_feature_table) != 200:
        raise ValueError(
            f"Test case 수가 200이 아닙니다: {len(test_feature_table)}"
        )

    # sample_submission/test_index와 case key가 동일한지 확인
    context_case_keys = set(
        map(
            tuple,
            test_feature_table[["case_id", "station"]]
            .drop_duplicates()
            .to_numpy()
        )
    )
    sample_case_keys = set(
        map(
            tuple,
            sample_submission[["case_id", "station"]]
            .drop_duplicates()
            .to_numpy()
        )
    )

    if context_case_keys != sample_case_keys:
        raise ValueError(
            "test_context에서 복원한 200개 case와 sample_submission의 case key가 다릅니다."
        )

    print("Test Feature Table 200개 case 생성 완료")
    print(f"Train Base shape: {train_feature_table.shape}")
    print(f"Test Base shape : {test_feature_table.shape}")
    print("[완료] Base Feature Table 복원 완료")

except Exception as e:
    print(f"[실패] Base Feature 생성 중 오류 발생: {e}")
    raise


Train Candidate 24,360개 정확 복원 완료
Train Feature Table 생성 완료
Test Feature Table 200개 case 생성 완료
Train Base shape: (24360, 219)
Test Base shape : (200, 211)
[완료] Base Feature Table 복원 완료


In [5]:
# ==== Final CoreAtmosCompact 130 Feature Schema ==== #

try:
    # 1) Wave Feature Set = Hs 55 + Wave Direction 11
    WAVE_FEATURES = list(
        dict.fromkeys(
            list(HS_FEATURES) + list(WVDIR_FEATURES)
        )
    )

    # 2) Compact Wind Magnitude — 25개
    COMPACT_WIND_MAG = [
        "wspd_current",
        "gust_current",
        "gust_excess_current",
        "wspd_lag_3h",
        "wspd_lag_6h",
        "wspd_lag_12h",
        "gust_lag_3h",
        "gust_lag_6h",
        "gust_lag_12h",
        "wspd_change_3h",
        "wspd_change_6h",
        "wspd_change_12h",
        "gust_change_3h",
        "gust_change_6h",
        "gust_change_12h",
        "wspd_mean_3h",
        "wspd_mean_6h",
        "wspd_mean_12h",
        "wspd_mean_24h",
        "gust_mean_3h",
        "gust_mean_6h",
        "gust_mean_12h",
        "gust_mean_24h",
        "wspd_valid_ratio_24h",
        "gust_valid_ratio_24h",
    ]

    # 3) Compact Wind Direction — 11개
    COMPACT_WIND_DIR = [
        "wdir_sin_current",
        "wdir_cos_current",
        "wdir_sin_lag_3h",
        "wdir_cos_lag_3h",
        "wdir_sin_lag_6h",
        "wdir_cos_lag_6h",
        "wdir_sin_lag_12h",
        "wdir_cos_lag_12h",
        "wdir_alignment_3h",
        "wdir_alignment_6h",
        "wdir_alignment_12h",
    ]

    # 4) Compact Wind-Wave Interaction — 12개
    COMPACT_WIND_WAVE = [
        "wind_wave_alignment_current",
        "wind_wave_cross_current",
        "wspd_parallel_wave_current",
        "gust_parallel_wave_current",
        "wind_wave_alignment_3h",
        "wind_wave_cross_3h",
        "wspd_parallel_wave_3h",
        "gust_parallel_wave_3h",
        "wind_wave_alignment_6h",
        "wind_wave_cross_6h",
        "wspd_parallel_wave_6h",
        "gust_parallel_wave_6h",
    ]

    # 5) Compact Pressure — 16개
    COMPACT_PRESSURE = [
        "caph_current",
        "caph_lag_3h",
        "caph_lag_6h",
        "caph_lag_12h",
        "caph_lag_24h",
        "caph_change_3h",
        "caph_change_6h",
        "caph_change_12h",
        "caph_change_24h",
        "caph_mean_6h",
        "caph_mean_12h",
        "caph_mean_24h",
        "caph_std_6h",
        "caph_std_12h",
        "caph_std_24h",
        "caph_valid_ratio_24h",
    ]

    CORE130_FEATURES = list(
        dict.fromkeys(
            WAVE_FEATURES
            + COMPACT_WIND_MAG
            + COMPACT_WIND_DIR
            + COMPACT_WIND_WAVE
            + COMPACT_PRESSURE
        )
    )

    if len(WAVE_FEATURES) != 66:
        raise ValueError(
            f"Wave Feature Set Feature 수 오류: {len(WAVE_FEATURES)}"
        )

    if len(CORE130_FEATURES) != 130:
        raise ValueError(
            f"CoreAtmosCompact Feature 수가 130이 아닙니다: {len(CORE130_FEATURES)}"
        )

    train_missing = [
        feature
        for feature in CORE130_FEATURES
        if feature not in train_feature_table.columns
    ]
    test_missing = [
        feature
        for feature in CORE130_FEATURES
        if feature not in test_feature_table.columns
    ]

    if train_missing or test_missing:
        raise KeyError(
            "Core130 Feature 누락 발생\n"
            f"Train: {train_missing}\n"
            f"Test : {test_missing}"
        )

    print("Feature groups")
    print(f"  Wave Feature Set        : {len(WAVE_FEATURES)}")
    print(f"  Compact Wind Mag     : {len(COMPACT_WIND_MAG)}")
    print(f"  Compact Wind Dir     : {len(COMPACT_WIND_DIR)}")
    print(f"  Compact Wind-Wave    : {len(COMPACT_WIND_WAVE)}")
    print(f"  Compact Pressure     : {len(COMPACT_PRESSURE)}")
    print(f"  Final Core130        : {len(CORE130_FEATURES)}")
    print("[완료] CoreAtmosCompact 130 Feature Schema 복원 완료")

except Exception as e:
    print(f"[실패] Core130 Schema 복원 중 오류 발생: {e}")
    raise


Feature groups
  Wave Feature Set        : 66
  Compact Wind Mag     : 25
  Compact Wind Dir     : 11
  Compact Wind-Wave    : 12
  Compact Pressure     : 16
  Final Core130        : 130
[완료] CoreAtmosCompact 130 Feature Schema 복원 완료


In [6]:
# ==== AtmosShape 19 Feature 생성 함수 정의 ==== #

def window_points(
    hours,
    step_minutes
):
    """
    지정 시간창에 포함되는 grid point 수를 계산한다.

    예:
    Wave 20분 grid에서 3h
    → 3×60/20 + 1 = 10 points
    """

    return (
        int(
            round(
                hours
                * 60
                / step_minutes
            )
        )
        + 1
    )

def rolling_slope(
    series,
    step_minutes,
    hours,
    min_valid_ratio=0.5
):
    """
    고정 시간격자에서 rolling linear regression slope를 계산한다.

    단위:
        series 단위 / hour

    결측이 존재해도 실제 유효 관측만 이용한다.
    """

    y = pd.to_numeric(
        series,
        errors="coerce"
    ).astype(float)


    n_window = (
        window_points(
            hours,
            step_minutes
        )
    )


    min_valid = max(
        3,
        int(
            np.ceil(
                n_window
                * min_valid_ratio
            )
        )
    )


    x = (
        np.arange(
            len(y),
            dtype=float
        )
        * step_minutes
        / 60.0
    )


    valid = (
        y.notna()
        .astype(float)
    )


    y0 = (
        y.fillna(0.0)
    )


    x_series = pd.Series(
        x,
        index=y.index
    )


    sum_n = (
        valid
        .rolling(
            n_window,
            min_periods=1
        )
        .sum()
    )


    sum_x = (
        (
            x_series
            * valid
        )
        .rolling(
            n_window,
            min_periods=1
        )
        .sum()
    )


    sum_y = (
        y0
        .rolling(
            n_window,
            min_periods=1
        )
        .sum()
    )


    sum_x2 = (
        (
            x_series
            * x_series
            * valid
        )
        .rolling(
            n_window,
            min_periods=1
        )
        .sum()
    )


    sum_xy = (
        (
            x_series
            * y0
        )
        .rolling(
            n_window,
            min_periods=1
        )
        .sum()
    )


    denominator = (

        sum_n
        * sum_x2

        -

        sum_x
        * sum_x
    )


    numerator = (

        sum_n
        * sum_xy

        -

        sum_x
        * sum_y
    )


    slope = (

        numerator

        /

        denominator.replace(
            0.0,
            np.nan
        )
    )


    slope[
        sum_n
        < min_valid
    ] = np.nan


    return slope

def ewm_gap(
    series,
    step_minutes,
    hours
):
    """
    현재값 - exponential moving mean.

    양수:
        최근 평균보다 현재값이 높은 상태.

    음수:
        최근 평균보다 낮은 상태.
    """

    y = pd.to_numeric(
        series,
        errors="coerce"
    ).astype(float)


    span = max(
        2,
        int(
            round(
                hours
                * 60
                / step_minutes
            )
        )
    )


    ewm_mean = (

        y
        .ewm(
            span=span,
            adjust=False,
            ignore_na=True
        )
        .mean()
    )


    return (
        y
        -
        ewm_mean
    )

def rolling_circular_concentration(
    degree_series,
    step_minutes,
    hours,
    min_valid_ratio=0.5
):
    """
    방향자료의 rolling mean resultant length R.

    R ≈ 1:
        최근 방향이 매우 일정함.

    R ≈ 0:
        최근 방향이 크게 변함.
    """

    degree = pd.to_numeric(
        degree_series,
        errors="coerce"
    ).astype(float)


    radians = np.deg2rad(
        degree
    )


    sin_series = pd.Series(
        np.sin(
            radians
        ),
        index=degree.index
    ).where(
        degree.notna()
    )


    cos_series = pd.Series(
        np.cos(
            radians
        ),
        index=degree.index
    ).where(
        degree.notna()
    )


    n_window = (
        window_points(
            hours,
            step_minutes
        )
    )


    min_valid = max(
        3,
        int(
            np.ceil(
                n_window
                * min_valid_ratio
            )
        )
    )


    sin_mean = (

        sin_series
        .rolling(
            n_window,
            min_periods=min_valid
        )
        .mean()
    )


    cos_mean = (

        cos_series
        .rolling(
            n_window,
            min_periods=min_valid
        )
        .mean()
    )


    return np.sqrt(
        sin_mean ** 2
        +
        cos_mean ** 2
    )

def build_atmos_shape_frame(
    station_atmos
):
    """
    10분 atmospheric sequence에서
    풍속/돌풍/기압/풍향 trajectory 정보를 추출한다.
    """

    g = (
        station_atmos
        .sort_values(
            "time"
        )
        .reset_index(drop=True)
        .copy()
    )


    out = (

        g[
            [
                "station",
                "time"
            ]
        ]
        .copy()
    )


    # -------------------------------------------------------------
    # 1. Wind speed trend
    # -------------------------------------------------------------

    for hours in [
        1,
        3,
        6,
        12,
        24
    ]:

        out[
            f"shape_wspd_slope_{hours}h"
        ] = (
            rolling_slope(
                g[
                    "wspd"
                ],
                10,
                hours
            )
        )


    # -------------------------------------------------------------
    # 2. Gust trend
    # -------------------------------------------------------------

    for hours in [
        3,
        6,
        12
    ]:

        out[
            f"shape_gust_slope_{hours}h"
        ] = (
            rolling_slope(
                g[
                    "gust"
                ],
                10,
                hours
            )
        )


    # -------------------------------------------------------------
    # 3. Pressure tendency
    # -------------------------------------------------------------

    for hours in [
        3,
        6,
        12,
        24
    ]:

        out[
            f"shape_caph_slope_{hours}h"
        ] = (
            rolling_slope(
                g[
                    "caph"
                ],
                10,
                hours
            )
        )


    # -------------------------------------------------------------
    # 4. Wind direction persistence
    # -------------------------------------------------------------

    for hours in [
        3,
        6,
        12,
        24
    ]:

        out[
            f"shape_wdir_concentration_{hours}h"
        ] = (
            rolling_circular_concentration(
                g[
                    "wdir"
                ],
                10,
                hours
            )
        )


    # -------------------------------------------------------------
    # 5. Wind speed EWM anomaly
    # -------------------------------------------------------------

    for hours in [
        3,
        6,
        12
    ]:

        out[
            f"shape_wspd_ewmgap_{hours}h"
        ] = (
            ewm_gap(
                g[
                    "wspd"
                ],
                10,
                hours
            )
        )


    return out

# 기존 검증 과정과 동일한 방식으로 Feature 이름/순서를 자동 복원한다.
_dummy_atmos = pd.DataFrame({
    "station": ["X"] * 300,
    "time": pd.date_range("2000-01-01", periods=300, freq="10min"),
    "wspd": np.ones(300),
    "gust": np.ones(300),
    "wdir": np.ones(300),
    "airt": np.ones(300),
    "relh": np.ones(300),
    "caph": np.ones(300),
})

_dummy_atmos_shape = build_atmos_shape_frame(_dummy_atmos)

ATMOS_SHAPE_FEATURES = [
    col
    for col in _dummy_atmos_shape.columns
    if col not in ["station", "time"]
]

del _dummy_atmos
del _dummy_atmos_shape

if len(ATMOS_SHAPE_FEATURES) != 19:
    raise ValueError(
        f"AtmosShape Feature 수가 19가 아닙니다: {len(ATMOS_SHAPE_FEATURES)}"
    )

print(f"AtmosShape Features: {len(ATMOS_SHAPE_FEATURES)}")
print("[완료] AtmosShape Feature Builder 정의 완료")


AtmosShape Features: 19
[완료] AtmosShape Feature Builder 정의 완료


In [7]:
# ==== Train/Test AtmosShape Table 및 Final Atmos149 Schema 생성 ==== #

try:
    # -----------------------------------------------------------------
    # Train: station별 연속 10분 atmosphere sequence에서 shape 계산
    # -----------------------------------------------------------------
    atmos_shape_frames = []

    for station in STATION_ORDER:
        print(f"[Train AtmosShape] {station}")

        station_atmos = (
            train_atmos[
                train_atmos["station"].eq(station)
            ]
            .copy()
        )

        atmos_shape_frames.append(
            build_atmos_shape_frame(station_atmos)
        )

    train_atmos_shape = pd.concat(
        atmos_shape_frames,
        ignore_index=True,
    )

    train_atmos_table = (
        train_feature_table
        .merge(
            train_atmos_shape[
                ["station", "time"] + ATMOS_SHAPE_FEATURES
            ],
            on=["station", "time"],
            how="left",
            validate="one_to_one",
        )
    )

    print(
        f"Train AtmosShape merge 완료: "
        f"{train_atmos_table.shape}"
    )


    # -----------------------------------------------------------------
    # Test: 각 case의 -48h ~ 0h context를 독립 sequence로 계산
    # -----------------------------------------------------------------
    test_atmos_shape_rows = []

    case_ids = (
        test_context["case_id"]
        .drop_duplicates()
        .tolist()
    )

    for case_index, case_id in enumerate(case_ids, start=1):
        if (
            case_index == 1
            or case_index % 20 == 0
            or case_index == len(case_ids)
        ):
            print(f"[Test AtmosShape {case_index}/{len(case_ids)}] {case_id}")

        case_df = (
            test_context[
                test_context["case_id"].eq(case_id)
            ]
            .sort_values("step_minute")
            .copy()
        )

        station = case_df["station"].iloc[0]

        # 기존 검증 과정과 동일하게 상대시간을 임시 datetime으로 변환한다.
        base_time = pd.Timestamp("2000-01-03 00:00:00")
        case_df["time"] = (
            base_time
            + pd.to_timedelta(
                case_df["step_minute"],
                unit="m",
            )
        )

        atmos_case = (
            case_df[
                [
                    "station",
                    "time",
                    "wspd",
                    "gust",
                    "wdir",
                    "airt",
                    "relh",
                    "caph",
                ]
            ]
            .copy()
        )

        atmos_shape = build_atmos_shape_frame(atmos_case)

        row = {
            "case_id": case_id,
            "station": station,
        }

        for feature in ATMOS_SHAPE_FEATURES:
            row[feature] = atmos_shape[feature].iloc[-1]

        test_atmos_shape_rows.append(row)

    test_atmos_shape = pd.DataFrame(test_atmos_shape_rows)

    if len(test_atmos_shape) != 200:
        raise ValueError(
            f"Test AtmosShape case 수가 200이 아닙니다: {len(test_atmos_shape)}"
        )

    test_atmos_table = (
        test_feature_table
        .merge(
            test_atmos_shape,
            on=["case_id", "station"],
            how="left",
            validate="one_to_one",
        )
    )

    print(
        f"Test AtmosShape merge 완료: "
        f"{test_atmos_table.shape}"
    )


    # -----------------------------------------------------------------
    # 최종 +3h branch = Core130 + AtmosShape19 = Atmos149
    # -----------------------------------------------------------------
    ATMOS149_FEATURES = list(
        dict.fromkeys(
            CORE130_FEATURES
            + ATMOS_SHAPE_FEATURES
        )
    )

    if len(ATMOS149_FEATURES) != 149:
        raise ValueError(
            f"Atmos149 Feature 수가 149가 아닙니다: {len(ATMOS149_FEATURES)}"
        )

    train_missing = [
        feature
        for feature in ATMOS149_FEATURES
        if feature not in train_atmos_table.columns
    ]
    test_missing = [
        feature
        for feature in ATMOS149_FEATURES
        if feature not in test_atmos_table.columns
    ]

    if train_missing or test_missing:
        raise KeyError(
            "Atmos149 Feature 누락 발생\n"
            f"Train: {train_missing}\n"
            f"Test : {test_missing}"
        )

    print(f"Core130 Feature 수  : {len(CORE130_FEATURES)}")
    print(f"AtmosShape 추가 수 : {len(ATMOS_SHAPE_FEATURES)}")
    print(f"Atmos149 Feature 수: {len(ATMOS149_FEATURES)}")
    print("[완료] Atmos149 Train/Test Table 생성 완료")

except Exception as e:
    print(f"[실패] AtmosShape Table 생성 중 오류 발생: {e}")
    raise


[Train AtmosShape] G-ORS
[Train AtmosShape] I-ORS
[Train AtmosShape] S-ORS
Train AtmosShape merge 완료: (24360, 238)
[Test AtmosShape 1/200] C0001
[Test AtmosShape 20/200] C0020
[Test AtmosShape 40/200] C0040
[Test AtmosShape 60/200] C0060
[Test AtmosShape 80/200] C0080
[Test AtmosShape 100/200] C0100
[Test AtmosShape 120/200] C0120
[Test AtmosShape 140/200] C0140
[Test AtmosShape 160/200] C0160
[Test AtmosShape 180/200] C0180
[Test AtmosShape 200/200] C0200
Test AtmosShape merge 완료: (200, 230)
Core130 Feature 수  : 130
AtmosShape 추가 수 : 19
Atmos149 Feature 수: 149
[완료] Atmos149 Train/Test Table 생성 완료


In [8]:
# ==== Final Model Matrix / Target / CatBoost Recipe 정의 ==== #

def make_station_onehot(df):
    """
    G-ORS / I-ORS / S-ORS 순서의 Station One-Hot을 생성한다.
    """
    return np.column_stack([
        df["station"]
        .eq(station)
        .astype(float)
        .to_numpy()
        for station in STATION_ORDER
    ])


def prepare_global_matrix(df, feature_columns):
    """
    Numeric Feature 뒤에 Station One-Hot 3개를 결합한다.
    NaN은 CatBoost가 내부적으로 처리한다.
    """
    x_numeric = (
        df[feature_columns]
        .to_numpy(dtype=float)
    )
    x_station = make_station_onehot(df)

    x = np.column_stack([
        x_numeric,
        x_station,
    ])

    if np.isinf(x).any():
        raise ValueError("Model matrix에 Inf 값이 존재합니다.")

    return x


def get_delta_target(df, lead_h):
    """
    Residual target:
        ΔHs = Hs(t + lead) - Hs(t)
    """
    return (
        df[f"hs_lead_{lead_h}h"].to_numpy(dtype=float)
        - df["hs_current"].to_numpy(dtype=float)
    )


def delta_to_hs(df, delta_prediction):
    """
    ΔHs prediction을 절대 유의파고 Hs prediction으로 복원한다.
    """
    return np.clip(
        df["hs_current"].to_numpy(dtype=float)
        + np.asarray(delta_prediction, dtype=float),
        0.0,
        30.0,
    )


def make_catboost_params(seed):
    """
    Final Model에서 사용한 Depth-4 CatBoost recipe에
    고정 MultiSeed를 적용한다.
    """
    params = CATBOOST_BASE_PARAMS.copy()
    params["random_seed"] = int(seed)
    return params


try:
    X_CORE_TRAIN = prepare_global_matrix(
        train_feature_table,
        CORE130_FEATURES,
    )
    X_CORE_TEST = prepare_global_matrix(
        test_feature_table,
        CORE130_FEATURES,
    )

    X_ATMOS_TRAIN = prepare_global_matrix(
        train_atmos_table,
        ATMOS149_FEATURES,
    )
    X_ATMOS_TEST = prepare_global_matrix(
        test_atmos_table,
        ATMOS149_FEATURES,
    )

    if X_CORE_TRAIN.shape != (24360, 133):
        raise ValueError(
            f"Core X_train shape 오류: {X_CORE_TRAIN.shape}"
        )
    if X_CORE_TEST.shape != (200, 133):
        raise ValueError(
            f"Core X_test shape 오류: {X_CORE_TEST.shape}"
        )
    if X_ATMOS_TRAIN.shape != (24360, 152):
        raise ValueError(
            f"Atmos X_train shape 오류: {X_ATMOS_TRAIN.shape}"
        )
    if X_ATMOS_TEST.shape != (200, 152):
        raise ValueError(
            f"Atmos X_test shape 오류: {X_ATMOS_TEST.shape}"
        )

    print(f"Core X_train  : {X_CORE_TRAIN.shape}")
    print(f"Core X_test   : {X_CORE_TEST.shape}")
    print(f"Atmos X_train : {X_ATMOS_TRAIN.shape}")
    print(f"Atmos X_test  : {X_ATMOS_TEST.shape}")
    print(f"Fixed Seeds   : {MULTISEED_LIST}")
    print("[완료] Final Model Matrix 및 Recipe 설정 완료")

except Exception as e:
    print(f"[실패] Model Matrix 생성 중 오류 발생: {e}")
    raise


Core X_train  : (24360, 133)
Core X_test   : (200, 133)
Atmos X_train : (24360, 152)
Atmos X_test  : (200, 152)
Fixed Seeds   : [11, 29, 47, 71, 101, 137, 173, 211, 251, 307]
[완료] Final Model Matrix 및 Recipe 설정 완료


In [9]:
# ==== 최종 60-Model Ensemble 학습 및 MultiSeed 평균 예측 ==== #

try:
    FINAL_MODELS = {
        "AtmosShape149": {
            3: {}
        },
        "Core130": {
            lead_h: {}
            for lead_h in [6, 9, 12, 18, 24]
        },
    }

    mean_delta_by_lead = {}


    # -----------------------------------------------------------------
    # +3h: AtmosShape149 × 10 seeds
    # -----------------------------------------------------------------
    print("=" * 120)
    print("FINAL TRAINING — +3h AtmosShape149 × MultiSeed10")
    print("=" * 120)

    y_atmos_3h = get_delta_target(
        train_atmos_table,
        3,
    )

    atmos_seed_predictions = []

    for seed_index, seed in enumerate(
        MULTISEED_LIST,
        start=1,
    ):
        print(
            f"[Atmos +3h | Seed {seed_index}/10] "
            f"{seed}"
        )

        model = CatBoostRegressor(
            **make_catboost_params(seed)
        )
        model.fit(
            X_ATMOS_TRAIN,
            y_atmos_3h,
        )

        delta_pred = np.asarray(
            model.predict(X_ATMOS_TEST),
            dtype=float,
        )

        if delta_pred.shape != (200,):
            raise ValueError(
                f"Atmos +3h Seed {seed} prediction shape 오류: "
                f"{delta_pred.shape}"
            )

        FINAL_MODELS["AtmosShape149"][3][seed] = model
        atmos_seed_predictions.append(delta_pred)

    atmos_seed_matrix = np.vstack(
        atmos_seed_predictions
    )
    mean_delta_by_lead[3] = np.mean(
        atmos_seed_matrix,
        axis=0,
    )

    print(
        f"[+3h 완료] Seed matrix = "
        f"{atmos_seed_matrix.shape}"
    )


    # -----------------------------------------------------------------
    # +6/+9/+12/+18/+24h: Core130 × 10 seeds
    # -----------------------------------------------------------------
    print("\n" + "=" * 120)
    print("FINAL TRAINING — Core130 × MultiSeed10")
    print("=" * 120)

    core_leads = [6, 9, 12, 18, 24]
    total_core_models = (
        len(core_leads)
        * len(MULTISEED_LIST)
    )
    model_counter = 0

    for lead_h in core_leads:
        y_train = get_delta_target(
            train_feature_table,
            lead_h,
        )

        lead_predictions = []

        print(f"\n[Core +{lead_h}h]")

        for seed_index, seed in enumerate(
            MULTISEED_LIST,
            start=1,
        ):
            model_counter += 1

            print(
                f"[Core {model_counter}/{total_core_models}] "
                f"+{lead_h}h | Seed {seed_index}/10 = {seed}"
            )

            model = CatBoostRegressor(
                **make_catboost_params(seed)
            )
            model.fit(
                X_CORE_TRAIN,
                y_train,
            )

            delta_pred = np.asarray(
                model.predict(X_CORE_TEST),
                dtype=float,
            )

            if delta_pred.shape != (200,):
                raise ValueError(
                    f"Core +{lead_h}h Seed {seed} prediction shape 오류: "
                    f"{delta_pred.shape}"
                )

            FINAL_MODELS["Core130"][lead_h][seed] = model
            lead_predictions.append(delta_pred)

        lead_matrix = np.vstack(
            lead_predictions
        )
        mean_delta_by_lead[lead_h] = np.mean(
            lead_matrix,
            axis=0,
        )

        print(
            f"[Core +{lead_h}h 완료] "
            f"Seed matrix = {lead_matrix.shape}"
        )

    trained_model_count = (
        len(FINAL_MODELS["AtmosShape149"][3])
        + sum(
            len(seed_models)
            for seed_models in FINAL_MODELS["Core130"].values()
        )
    )

    if trained_model_count != 60:
        raise ValueError(
            f"최종 학습 모델 수가 60개가 아닙니다: {trained_model_count}"
        )

    print(f"\nFinal trained models: {trained_model_count}")
    print("[완료] Final Model 60-Model 학습 완료")

except Exception as e:
    print(f"[실패] Final Model 학습 중 오류 발생: {e}")
    raise


FINAL TRAINING — +3h AtmosShape149 × MultiSeed10
[Atmos +3h | Seed 1/10] 11
[Atmos +3h | Seed 2/10] 29
[Atmos +3h | Seed 3/10] 47
[Atmos +3h | Seed 4/10] 71
[Atmos +3h | Seed 5/10] 101
[Atmos +3h | Seed 6/10] 137
[Atmos +3h | Seed 7/10] 173
[Atmos +3h | Seed 8/10] 211
[Atmos +3h | Seed 9/10] 251
[Atmos +3h | Seed 10/10] 307
[+3h 완료] Seed matrix = (10, 200)

FINAL TRAINING — Core130 × MultiSeed10

[Core +6h]
[Core 1/50] +6h | Seed 1/10 = 11
[Core 2/50] +6h | Seed 2/10 = 29
[Core 3/50] +6h | Seed 3/10 = 47
[Core 4/50] +6h | Seed 4/10 = 71
[Core 5/50] +6h | Seed 5/10 = 101
[Core 6/50] +6h | Seed 6/10 = 137
[Core 7/50] +6h | Seed 7/10 = 173
[Core 8/50] +6h | Seed 8/10 = 211
[Core 9/50] +6h | Seed 9/10 = 251
[Core 10/50] +6h | Seed 10/10 = 307
[Core +6h 완료] Seed matrix = (10, 200)

[Core +9h]
[Core 11/50] +9h | Seed 1/10 = 11
[Core 12/50] +9h | Seed 2/10 = 29
[Core 13/50] +9h | Seed 3/10 = 47
[Core 14/50] +9h | Seed 4/10 = 71
[Core 15/50] +9h | Seed 5/10 = 101
[Core 16/50] +9h | Seed 6/10 =

In [10]:
# ==== submission.csv / final_models.pkl / gate_params_final.pkl 생성 ==== #

def sha256_file(filepath):
    """
    파일의 SHA256 checksum을 계산한다.
    """
    digest = hashlib.sha256()

    with open(filepath, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


try:
    # -----------------------------------------------------------------
    # 200 cases × 6 leads prediction long table 생성
    # -----------------------------------------------------------------
    prediction_frames = []

    for lead_h in LEAD_HOURS:
        hs_pred = delta_to_hs(
            test_feature_table,
            mean_delta_by_lead[lead_h],
        )

        prediction_frames.append(
            pd.DataFrame({
                "case_id": test_feature_table["case_id"].to_numpy(),
                "station": test_feature_table["station"].to_numpy(),
                "lead_h": lead_h,
                "hs_pred": hs_pred,
            })
        )

    prediction_long = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    if len(prediction_long) != 1200:
        raise ValueError(
            f"Prediction row 수가 1,200이 아닙니다: {len(prediction_long)}"
        )


    # -----------------------------------------------------------------
    # sample_submission의 key/order를 보존
    # -----------------------------------------------------------------
    submission = (
        sample_submission[KEY_COLUMNS]
        .merge(
            prediction_long,
            on=KEY_COLUMNS,
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    if len(submission) != len(sample_submission):
        raise ValueError("Submission row 수가 sample_submission과 다릅니다.")

    if submission["hs_pred"].isna().any():
        raise ValueError("Submission prediction에 NaN이 존재합니다.")

    if not np.isfinite(
        submission["hs_pred"].to_numpy(dtype=float)
    ).all():
        raise ValueError("Submission prediction에 Inf가 존재합니다.")

    if (
        submission[KEY_COLUMNS]
        .reset_index(drop=True)
        .equals(
            sample_submission[KEY_COLUMNS]
            .reset_index(drop=True)
        )
        is False
    ):
        raise ValueError(
            "Submission key order가 sample_submission과 다릅니다."
        )

    if submission.duplicated(KEY_COLUMNS).any():
        raise ValueError("Submission key에 중복이 존재합니다.")

    submission.to_csv(
        SUBMISSION_FILE,
        index=False,
        encoding="utf-8-sig",
    )


    # -----------------------------------------------------------------
    # final_models.pkl
    # 최종 모델 prediction에 실제로 사용되는 60개 CatBoost만 저장한다.
    # -----------------------------------------------------------------
    final_models_payload = {
        "version": "P3_FINAL_MODEL_V1",
        "model_name": "Hybrid_Atmos03h_CoreOther_MultiSeed10",
        "models": FINAL_MODELS,
        "metadata": {
            "lead_hours": LEAD_HOURS,
            "station_order": STATION_ORDER,
            "seeds": MULTISEED_LIST,
            "target": "delta_hs",
            "aggregation": "arithmetic_mean_across_10_seeds",
            "prediction_clip_m": [0.0, 30.0],
            "catboost_version": catboost.__version__,
            "catboost_base_params": CATBOOST_BASE_PARAMS,
            "core_feature_count": len(CORE130_FEATURES),
            "atmos_feature_count": len(ATMOS149_FEATURES),
        },
    }

    with open(
        FINAL_MODELS_FILE,
        "wb",
    ) as f:
        pickle.dump(
            final_models_payload,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )


    # -----------------------------------------------------------------
    # gate_params_final.pkl
    # 고정 branch routing/configuration metadata.
    # 파일명은 제출 repository 구조와의 호환성을 위해 유지한다.
    # -----------------------------------------------------------------
    gate_params_final = {
        "version": "P3_FINAL_ROUTING_V1",
        "routing": {
            3: "AtmosShape149",
            6: "Core130",
            9: "Core130",
            12: "Core130",
            18: "Core130",
            24: "Core130",
        },
        "seeds": MULTISEED_LIST,
        "station_order": STATION_ORDER,
        "target": "delta_hs",
        "aggregation": "arithmetic_mean",
        "prediction_clip_m": [0.0, 30.0],
        "core_features": CORE130_FEATURES,
        "atmos_shape_features": ATMOS_SHAPE_FEATURES,
        "atmos149_features": ATMOS149_FEATURES,
        "catboost_base_params": CATBOOST_BASE_PARAMS,
    }

    with open(
        GATE_PARAMS_FILE,
        "wb",
    ) as f:
        pickle.dump(
            gate_params_final,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )


    # -----------------------------------------------------------------
    # 최종 파일 무결성 및 검증 기준 fingerprint 확인
    # -----------------------------------------------------------------
    submission_hash = sha256_file(
        SUBMISSION_FILE
    )

    validated_submission_sha256 = (
        "59da21d052e08255473461dac00078b477c31a902afb7f63f1f79930dca2f851"
    )

    print("=" * 120)
    print("FINAL ARTIFACTS")
    print("=" * 120)

    for filepath in [
        FINAL_MODELS_FILE,
        GATE_PARAMS_FILE,
        SUBMISSION_FILE,
    ]:
        size_mb = filepath.stat().st_size / (1024 ** 2)
        print(
            f"{filepath.name:<24} "
            f"{size_mb:>8.3f} MB"
        )

    print("\nSubmission summary")
    print(f"Rows       : {len(submission):,}")
    print(f"Min Hs     : {submission['hs_pred'].min():.6f} m")
    print(f"Mean Hs    : {submission['hs_pred'].mean():.6f} m")
    print(f"Max Hs     : {submission['hs_pred'].max():.6f} m")
    print(f"SHA256     : {submission_hash}")
    print(f"Validated  : {validated_submission_sha256}")

    if submission_hash == validated_submission_sha256:
        print(
            "[검증 성공] 검증 기준 submission과 "
            "CSV SHA256까지 정확히 일치합니다."
        )
    else:
        print(
            "[주의] 검증 기준 CSV SHA256과 다릅니다. "
            "CatBoost/Pandas 버전 또는 실행환경 차이를 우선 확인하세요."
        )

    print("[완료] 최종 제출 파일 3종 생성 완료")

except Exception as e:
    print(f"[실패] 최종 파일 생성 중 오류 발생: {e}")
    raise


FINAL ARTIFACTS
final_models.pkl            9.359 MB
gate_params_final.pkl       0.003 MB
submission.csv              0.039 MB

Submission summary
Rows       : 1,200
Min Hs     : 0.590841 m
Mean Hs    : 1.693258 m
Max Hs     : 4.238339 m
SHA256     : 59da21d052e08255473461dac00078b477c31a902afb7f63f1f79930dca2f851
Validated  : 59da21d052e08255473461dac00078b477c31a902afb7f63f1f79930dca2f851
[검증 성공] 검증 기준 submission과 CSV SHA256까지 정확히 일치합니다.
[완료] 최종 제출 파일 3종 생성 완료


In [11]:
# ==== 저장된 PKL 재로드 및 Inference 재현성 확인 ==== #

try:
    with open(FINAL_MODELS_FILE, "rb") as f:
        reloaded_models = pickle.load(f)

    with open(GATE_PARAMS_FILE, "rb") as f:
        reloaded_gate = pickle.load(f)

    reloaded_delta_by_lead = {}

    for lead_h in LEAD_HOURS:
        branch = reloaded_gate["routing"][lead_h]

        if branch == "AtmosShape149":
            x_test = X_ATMOS_TEST
            seed_models = (
                reloaded_models["models"]
                ["AtmosShape149"]
                [lead_h]
            )
        elif branch == "Core130":
            x_test = X_CORE_TEST
            seed_models = (
                reloaded_models["models"]
                ["Core130"]
                [lead_h]
            )
        else:
            raise ValueError(
                f"알 수 없는 routing branch: {branch}"
            )

        seed_predictions = []

        for seed in reloaded_gate["seeds"]:
            model = seed_models[seed]

            seed_predictions.append(
                np.asarray(
                    model.predict(x_test),
                    dtype=float,
                )
            )

        reloaded_delta_by_lead[lead_h] = (
            np.mean(
                np.vstack(seed_predictions),
                axis=0,
            )
        )

    reload_frames = []

    for lead_h in LEAD_HOURS:
        reload_frames.append(
            pd.DataFrame({
                "case_id": test_feature_table["case_id"].to_numpy(),
                "station": test_feature_table["station"].to_numpy(),
                "lead_h": lead_h,
                "hs_pred_reload": delta_to_hs(
                    test_feature_table,
                    reloaded_delta_by_lead[lead_h],
                ),
            })
        )

    reload_long = pd.concat(
        reload_frames,
        ignore_index=True,
    )

    reload_check = (
        submission
        .merge(
            reload_long,
            on=KEY_COLUMNS,
            how="left",
            validate="one_to_one",
        )
    )

    reload_check["abs_diff"] = np.abs(
        reload_check["hs_pred"]
        - reload_check["hs_pred_reload"]
    )

    max_reload_diff = float(
        reload_check["abs_diff"].max()
    )

    if max_reload_diff > 1e-12:
        raise ValueError(
            "저장된 final_models.pkl 재로드 prediction이 "
            f"submission과 일치하지 않습니다: max diff={max_reload_diff}"
        )

    print(f"Reload max absolute difference: {max_reload_diff:.12e} m")
    print("[완료] 저장 Model Artifact inference 재현성 검증 완료")

except Exception as e:
    print(f"[실패] PKL 재로드 검증 중 오류 발생: {e}")
    raise


Reload max absolute difference: 0.000000000000e+00 m
[완료] 저장 Model Artifact inference 재현성 검증 완료


## Expected repository output

Notebook 실행이 완료되면 저장소 루트는 다음 구조가 된다.

```text
P3/
├─ data/
│  ├─ train_wave.csv
│  ├─ train_atmos.csv
│  ├─ test_context.parquet
│  ├─ test_index.csv
│  └─ sample_submission.csv
├─ pipeline_final.ipynb
├─ final_models.pkl
├─ gate_params_final.pkl
└─ submission.csv
```
